# Đồ án 3: Gram-Schmidt QR Decomposition


**Môn:** Toán ứng dụng và thống kê cho Công nghệ thông tin


**Sinh viên:** Nguyễn Thế Hiển




**MSSV:** 22127107




**Lớp:** 24C05




## 1. Mở đầu


QR decomposition là một trong những phân rã quan trọng nhất trong đại số tuyến tính ứng dụng. Trong đồ án này, ta tự cài đặt thuật toán Gram-Schmidt để phân rã ma trận $A$ thành hai ma trận $Q$ và $R$ sao cho $A = QR$. Mục tiêu là hiểu rõ cơ chế tạo ra các vector trực chuẩn, kiểm tra tính đúng đắn của kết quả, và liên hệ với các ứng dụng thực tế trong tính toán số.


## 2. Mục tiêu và phạm vi thực hiện


- Tự cài đặt thuật toán Gram-Schmidt để tìm $Q$ và $R$.
- Không dùng hàm QR có sẵn của thư viện để phân rã trực tiếp.
- So sánh kết quả với `numpy.linalg.qr` để đối chiếu.
- Trình bày ứng dụng, nhận xét và kết luận từ kết quả thực nghiệm.



## 3. Quy ước ký hiệu


| Ký hiệu | Ý nghĩa |


|---|---|


| $A$ | Ma trận đầu vào cần phân rã |


| $Q$ | Ma trận có các cột trực chuẩn |


| $R$ | Ma trận tam giác trên |


| $a_k$ | Cột thứ $k$ của ma trận $A$ |


| $q_k$ | Vector trực chuẩn thứ $k$ |


| $u_k$ | Vector trung gian sau khi khử chiếu |




Bảng trên dùng thống nhất các ký hiệu xuất hiện trong phần lý thuyết và phần cài đặt.


## 4. Cấu trúc nội dung


1. Trình bày ý tưởng Gram-Schmidt.

2. Cài đặt thuật toán phân rã QR.

3. Kiểm tra bằng ví dụ minh họa.

4. So sánh với kết quả từ thư viện.

5. Phân tích ứng dụng, độ phức tạp và nhận xét tổng kết.

## 5. Ý tưởng Gram-Schmidt


Giả sử các cột của ma trận $A$ là $a_1, a_2, \dots, a_n$. Ta xây dựng một hệ vector trực chuẩn $q_1, q_2, \dots, q_n$ theo nguyên tắc loại bỏ dần phần chiếu của mỗi vector mới lên các vector đã có trước đó.


Với mỗi cột thứ $k$ của $A$, ta thực hiện:


$$


u_k = a_k - \sum_{j=1}^{k-1} \operatorname{proj}_{q_j}(a_k),


\qquad


\operatorname{proj}_{q_j}(a_k) = (q_j^T a_k)q_j,


$$


sau đó chuẩn hóa để thu được


$$


q_k = \frac{u_k}{\lVert u_k \rVert}.


$$


Khi đã có ma trận $Q = [q_1, q_2, \dots, q_n]$, ta tính `R = Q^T A`. Vì các vector trong $Q$ trực chuẩn nên $Q^T Q = I$, do đó suy ra được $A = QR$.


### Nhận xét


- Nếu một cột của $A$ phụ thuộc tuyến tính vào các cột trước đó, thuật toán sẽ gặp trường hợp $\lVert u_k \rVert = 0$ và không thể tiếp tục theo cách này.
- Trong thực hành số học, Gram-Schmidt cổ điển có thể kém ổn định hơn các biến thể khác, nhưng trong phạm vi đồ án này nó phù hợp để minh họa ý tưởng QR decomposition.

In [13]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)

## 6. Cài đặt thuật toán Gram-Schmidt


Hàm bên dưới nhận một ma trận $A$ và trả về $Q$, $R$. Nếu phát hiện cột phụ thuộc tuyến tính gây chia cho 0, hàm sẽ báo lỗi rõ ràng.

In [14]:
def gram_schmidt_qr(A):
    A = np.array(A, dtype=float)
    m, n = A.shape
    Q = np.zeros((m, n), dtype=float)
    R = np.zeros((n, n), dtype=float)

    for k in range(n):
        v = A[:, k].copy()

        for j in range(k):
            R[j, k] = np.dot(Q[:, j], A[:, k])
            v = v - R[j, k] * Q[:, j]

        R[k, k] = np.linalg.norm(v)
        if np.isclose(R[k, k], 0.0):
            raise ValueError('Ma trận có cột phụ thuộc tuyến tính, không phân rã QR theo cách này.')

        Q[:, k] = v / R[k, k]

    return Q, R

## 7. Ví dụ minh họa


Ta chọn một ma trận vuông đơn giản để kiểm tra kết quả.

In [15]:
A = np.array([
    [1, 1, 0],
    [1, 0, 1],
    [0, 1, 1],
], dtype=float)

Q, R = gram_schmidt_qr(A)

print('A =')
print(A)
print('\nQ =')
print(Q)
print('\nR =')
print(R)

A =
[[1. 1. 0.]
 [1. 0. 1.]
 [0. 1. 1.]]

Q =
[[ 0.7071  0.4082 -0.5774]
 [ 0.7071 -0.4082  0.5774]
 [ 0.      0.8165  0.5774]]

R =
[[1.4142 0.7071 0.7071]
 [0.     1.2247 0.4082]
 [0.     0.     1.1547]]


In [16]:
reconstructed = Q @ R
orthogonality_check = Q.T @ Q

print('Q @ R =')
print(reconstructed)
print('\nQ^T @ Q =')
print(orthogonality_check)
print('\nA gần bằng Q @ R:', np.allclose(A, reconstructed))
print('Q trực chuẩn gần đúng:', np.allclose(orthogonality_check, np.eye(Q.shape[1])))

Q @ R =
[[1. 1. 0.]
 [1. 0. 1.]
 [0. 1. 1.]]

Q^T @ Q =
[[ 1.  0.  0.]
 [ 0.  1. -0.]
 [ 0. -0.  1.]]

A gần bằng Q @ R: True
Q trực chuẩn gần đúng: True


## 8. Ví dụ bổ sung


Để kiểm tra thuật toán trên một ma trận không vuông, ta xét thêm một ví dụ khác có 3 hàng và 2 cột. Ví dụ này cho thấy chương trình không chỉ hoạt động với ma trận vuông mà còn áp dụng được cho ma trận có số hàng lớn hơn số cột, miễn là các cột độc lập tuyến tính.

In [17]:
A2 = np.array([
    [2, 1],
    [1, 3],
    [0, 1],
])
Q2, R2 = gram_schmidt_qr(A2)

print('A2 =')
print(A2)
print('\nQ2 =')
print(Q2)
print('\nR2 =')
print(R2)
print('\nQ2 @ R2 =')
print(Q2 @ R2)
print('\nA2 gần bằng Q2 @ R2:', np.allclose(A2, Q2 @ R2))
print('Q2 trực chuẩn gần đúng:', np.allclose(Q2.T @ Q2, np.eye(Q2.shape[1])))

A2 =
[[2 1]
 [1 3]
 [0 1]]

Q2 =
[[ 0.8944 -0.4082]
 [ 0.4472  0.8165]
 [ 0.      0.4082]]

R2 =
[[2.2361 2.2361]
 [0.     2.4495]]

Q2 @ R2 =
[[2. 1.]
 [1. 3.]
 [0. 1.]]

A2 gần bằng Q2 @ R2: True
Q2 trực chuẩn gần đúng: True


### Nhận xét ví dụ bổ sung


Ví dụ này cho thấy thuật toán vẫn giữ được tính đúng đắn trên ma trận chữ nhật $3\times 2$. Kết quả `A2 gần bằng Q2 @ R2` là `True`, đồng thời `Q2^TQ2` gần bằng ma trận đơn vị, nên hệ vector tạo ra vẫn trực chuẩn.


So với ví dụ vuông ban đầu, ví dụ này giúp kiểm tra thêm rằng chương trình không bị phụ thuộc vào việc ma trận phải là ma trận vuông.

## 9. So sánh với thư viện


Để kiểm tra độ đúng đắn của kết quả, ta dùng `numpy.linalg.qr` làm mốc đối chiếu. Mục này không thay thế thuật toán tự cài đặt ở trên mà chỉ giúp xác nhận rằng phép phân rã thu được là hợp lý về mặt số học.


Khi so sánh, cần lưu ý rằng ma trận $Q$ và $R$ từ các phương pháp khác nhau có thể khác dấu ở một số cột, nhưng tích $Q R$ vẫn khôi phục lại ma trận gốc $A$.

In [18]:
Q_lib, R_lib = np.linalg.qr(A)

print('Q từ thư viện =')
print(Q_lib)
print('\nR từ thư viện =')
print(R_lib)
print('\nSai số |A - Q@R| =', np.linalg.norm(A - Q_lib @ R_lib))

Q từ thư viện =
[[-0.7071  0.4082 -0.5774]
 [-0.7071 -0.4082  0.5774]
 [-0.      0.8165  0.5774]]

R từ thư viện =
[[-1.4142 -0.7071 -0.7071]
 [ 0.      1.2247  0.4082]
 [ 0.      0.      1.1547]]

Sai số |A - Q@R| = 7.561852809663275e-16


## 10. Ứng dụng của QR decomposition


QR decomposition xuất hiện rất nhiều trong tính toán khoa học và xử lý dữ liệu. Một số ứng dụng tiêu biểu gồm:


- Giải hệ phương trình tuyến tính theo cách ổn định số hơn so với khử Gauss trong nhiều trường hợp.
- Bài toán bình phương tối thiểu (least squares), đặc biệt khi số phương trình lớn hơn số ẩn.
- Các thuật toán lặp để tìm trị riêng và phân tích ma trận.
- Hồi quy tuyến tính, xử lý tín hiệu và nhiều bài toán tối ưu trong khoa học dữ liệu.


Trong thực tế, QR decomposition là công cụ nền tảng để biến một bài toán phức tạp thành dạng dễ tính toán hơn, đồng thời giúp tăng độ ổn định số của quá trình giải.

## 11. Độ phức tạp và nhận xét


Với ma trận có $m$ hàng và $n$ cột, thuật toán Gram-Schmidt cổ điển có độ phức tạp xấp xỉ $O(mn^2)$. Điều này là hợp lý vì ở mỗi cột mới ta phải chiếu nó lên toàn bộ các vector trực chuẩn đã có trước đó.


Một số nhận xét quan trọng:


- Thuật toán rất phù hợp để minh họa ý tưởng QR decomposition trong đồ án học tập.
- Kết quả có thể khác dấu giữa các phương pháp khác nhau, nhưng tích $QR$ vẫn tái tạo được ma trận gốc.
- Trong tính toán thực tế quy mô lớn, các biến thể ổn định số hơn thường được ưu tiên hơn Gram-Schmidt cổ điển.


## 12. Kết luận


Đồ án đã tự cài đặt thành công thuật toán Gram-Schmidt để phân rã ma trận thành $Q$ và $R$ mà không dùng hàm QR có sẵn của thư viện để tính trực tiếp. Kết quả kiểm tra cho thấy $A \approx QR$ và $Q^TQ \approx I$, từ đó xác nhận thuật toán hoạt động đúng trên ví dụ minh họa đã chọn.


Phần so sánh với `numpy.linalg.qr` cho thấy kết quả thu được phù hợp về mặt số học, đồng thời giúp kiểm tra lại tính đúng đắn của chương trình.


## 13. Tài liệu tham khảo


- Giáo trình Toán ứng dụng và Thống kê trong CNTT.
- Nội dung lý thuyết Gram-Schmidt và QR decomposition trên lớp.
- Tài liệu `numpy.linalg.qr` của thư viện NumPy để đối chiếu kết quả.
- Ghi chú thực hành và bài tập trên lớp về phép phân rã QR.